In [1]:
import csv
import itertools
import os

def process_combination(combo, country_emissions):
    factors = [round(x / 10, 1) for x in range(0, 11)]

    sum_bau = sum(country_emissions[c][0] for c in combo)
    sum_lc  = sum(country_emissions[c][1] for c in combo)

    sum_bau_30 = sum_bau * 30
    sum_lc_30  = sum_lc * 30

    bau_factors = [sum_bau_30 * f for f in factors]

    less_equal_list = [bf for bf in bau_factors if bf <= sum_lc_30]
    greater_list    = [bf for bf in bau_factors if bf > sum_lc_30]

    final_values = []
    final_values.extend(less_equal_list)

    num_greater = len(greater_list)
    for i in range(1, num_greater + 1):
        new_val = (i * sum_lc_30) / num_greater
        final_values.append(new_val)

    final_values.sort()

    return final_values

def read_country_emissions():

    bau_path = os.path.join("..", "..", "Data", "Case_Study", "BAU_No_Action", "total_data_unrounded.csv")
    lc_path = os.path.join("..", "..", "Data", "Case_Study", "Least_Cost_Emissions", "total_data_unrounded.csv")

    def extract_first_emissions_per_country(file_path):
        emissions = {}
        seen = set()
        with open(file_path, mode='r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                country = row["country_1"].strip()
                if country and country not in seen:
                    try:
                        emissions[country] = float(row["collaboration_emissions"])
                        seen.add(country)
                    except ValueError:
                        print(f"⚠️ Could not parse emissions for {country} in {file_path}")
        return emissions

    bau_emissions = extract_first_emissions_per_country(bau_path)
    lc_emissions = extract_first_emissions_per_country(lc_path)

    all_countries = set(bau_emissions.keys()).union(lc_emissions.keys())

    country_emissions = {}
    for country in all_countries:
        bau = bau_emissions.get(country)
        lc = lc_emissions.get(country)

        if bau is None:
            print(f"⚠️ Country '{country}' not found in BAU emissions data.")
        if lc is None:
            print(f"⚠️ Country '{country}' not found in Least Cost emissions data.")

        if bau is not None and lc is not None:
            country_emissions[country] = (bau, lc)

    return country_emissions

def load_existing_cases(output_file):
    existing_cases = {}
    if os.path.exists(output_file):
        with open(output_file, mode='r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                case = row['case_study']
                if case not in existing_cases:
                    existing_cases[case] = []
                existing_cases[case].append(row['maximum_budget'])
    return existing_cases

def ensure_file_ends_with_newline(file_path):
    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
        with open(file_path, mode='rb+') as f:
            f.seek(-1, os.SEEK_END)
            last_char = f.read(1)
            if last_char != b'\n':
                f.write(b'\n')

def write_parameters(output_file, case_study, values, starting_type):
    ensure_file_ends_with_newline(output_file)  # 🛡 Ensure newline before appending

    with open(output_file, mode='a', newline='', encoding='utf-8') as out_f:
        writer = csv.writer(out_f)
        for val in values:
            formatted_val = f"{val:.2E}" if abs(val) >= 1000 else f"{val:.6f}"
            writer.writerow([starting_type, formatted_val, "MtCO2e", case_study])
            starting_type += 1
    return starting_type

def initialize_output_file(output_file):
    if not os.path.exists(output_file):
        with open(output_file, mode='w', newline='', encoding='utf-8') as out_f:
            writer = csv.writer(out_f)
            writer.writerow(["Type", "maximum_budget", "maximum_budget_unit", "case_study"])

def update_existing_case_values(output_file, case_study, new_values):
    if not os.path.exists(output_file):
        return

    with open(output_file, mode='r', encoding='utf-8') as f:
        rows = list(csv.reader(f))

    header = rows[0]
    updated_rows = []
    value_idx = 0

    for row in rows[1:]:
        if row[3] == case_study and value_idx < len(new_values):
            val = new_values[value_idx]
            formatted_val = f"{val:.2E}" if abs(val) >= 1000 else f"{val:.6f}"
            row[1] = formatted_val  # Replace only the maximum_budget
            value_idx += 1
        updated_rows.append(row)

    # If the case doesn't exist, append new rows
    if value_idx == 0:
        return False  # Signal that it was not updated (new case)

    with open(output_file, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(updated_rows)
    return True


def prompt_for_collaboration():
    resp = input("Do you want to generate parameters for a collaboration? (y/n): ").strip().lower()
    if resp.startswith("y"):
        combo = input("Enter comma-separated list of maximum 4 countries (e.g., ID,SG): ")
        return [c.strip().upper() for c in combo.split(",") if c.strip()]
    return None

def main():
    input_file = os.path.join("..", "..", "Data", "Case_Study", "input.csv")
    output_file = os.path.join("..", "Assets", "CO2_Budget", "parameters.csv")

    initialize_output_file(output_file)
    country_emissions = read_country_emissions()
    existing_cases = load_existing_cases(output_file)

    type_index = sum(len(v) for v in existing_cases.values())

    # Process all single-country cases
    for country in country_emissions:
        case_study = country
        #updated = update_existing_case_values(output_file, case_study, values)
        if case_study not in existing_cases or len(existing_cases[case_study]) < 11:
            print(f"Generating parameters for: {country}")
            values = process_combination([country], country_emissions)
            type_index = write_parameters(output_file, case_study, values, type_index)
        elif case_study in existing_cases:
             print(f"Updating parameters for: {country}")
             values = process_combination([country], country_emissions)
             updated = update_existing_case_values(output_file, case_study, values)
    
    # Ask if the user wants to add collaborations
    # while True:
    
    # combo = prompt_for_collaboration()
    combo = ["SG", "ID", "MY", "KH", "VN"]
    if not combo:
        pass
    if len(combo) > 4:
        print("⚠️ Please enter no more than 4 countries for collaboration.")
    else:
        # Generate all unique combinations of size 2 to len(combo)
        for r in range(2, len(combo) + 1):
            for sub_combo in itertools.combinations(combo, r):
                combo_sorted = sorted(sub_combo)
                case_study = "-".join(combo_sorted)

                if case_study in existing_cases:
                    print(f"Updating collaboration: {case_study}")
                    try:
                        values = process_combination(combo_sorted, country_emissions)
                        updated = update_existing_case_values(output_file, case_study, values)
                    except KeyError as e:
                        print(f"⚠️ Country not found in emissions data: {e}")
                else:
                    try:
                        print(f"Generating collaboration: {case_study}")
                        type_index = write_parameters(output_file, case_study, values, type_index)
                    except KeyError as e:
                        print(f"⚠️ Country not found in emissions data: {e}")

    print("✅ Done! Parameters written to parameters.csv")

In [2]:
import os
from pathlib import Path
os.chdir(Path(os.getcwd()).parent / "Code/Automations")

import random
import pandas as pd
pd.set_option('display.float_format', str)

In [3]:
input_file = os.path.join("..", "..", "Data", "Case_Study", "input.csv")
output_file = os.path.join("..", "Assets", "CO2_Budget", "parameters.csv")

In [4]:
initialize_output_file(output_file)
country_emissions = read_country_emissions()
existing_cases = load_existing_cases(output_file)

type_index = sum(len(v) for v in existing_cases.values())

In [5]:
country_emissions

{'AU': (231.7139038939378, 73.44161405029396),
 'RU': (836.3348290751843, 835.9482061518162),
 'PH': (144.71827874470037, 17.787553806831905),
 'PE': (14.09832051457294, 0.00309943767497316),
 'LA': (0.42689942685343946, 1.687588325391213e-05),
 'BR': (231.03336453257361, 20.082697238454905),
 'US': (2246.757983061586, 2235.4461768382926),
 'CO': (38.446625825141865, 1.576164530516447),
 'EG': (247.16774257524327, 22.63325870933477),
 'NG': (271.2364516608728, 54.521569650274856),
 'ZA': (232.32841034452818, 32.089379790483505),
 'JP': (638.9914528560747, 642.144581455264),
 'MA': (67.9415680802466, 7.8975253915336605),
 'VN': (254.96142416166452, 36.519385601215376),
 'MY': (157.61213311559212, 46.8818349189125),
 'FR': (49.522300228084596, 47.03704390157557),
 'ID': (703.6109178468002, 104.81347666725827),
 'KR': (301.976753841845, 98.47299058666297),
 'SG': (37.555003427011734, 36.79722395108159),
 'KH': (16.599016733355043, 3.741204373437601),
 'CL': (36.889862832337776, 1.25874256

In [6]:
dfs = list()
for key, val in zip(existing_cases.keys(), existing_cases.values()):
    dfs.append(pd.DataFrame({key: pd.Series(val)}))
data = pd.concat(dfs, axis=1)
data = data.astype(float)
rand_cols = random.sample(data.columns.tolist(), 10)
data[rand_cols]

,LA-TH-VN,BN-MY-PH-TH,FR-MA-TR,EG-KE,ID-MY-PH-SG,ID-LA,EG-KE-ZA,KH-SG,LA-TH,LA-SG
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,298.828412,458.187752,159.932975,76.898498,687.600292,349.378306,192.407375,162.46206,199.234268,113.943875
2,597.656825,916.375504,319.86595,153.796997,1380.0,698.756613,384.814751,324.924121,398.468536,227.88775
3,896.485237,1370.0,479.798924,230.695495,2060.0,1050.0,577.222126,405.384283,597.702804,341.831624
4,1200.0,1710.0,639.731899,307.593994,2750.0,1400.0,769.629502,487.386181,789.388088,455.775499
5,1490.0,1830.0,705.571534,384.492492,3130.0,1750.0,962.036877,649.848242,796.937072,569.719374
6,1550.0,2290.0,799.664874,461.390991,3440.0,2100.0,1150.0,810.768566,996.17134,683.663249
7,1790.0,2750.0,959.597849,538.289489,4130.0,2110.0,1350.0,812.310302,1200.0,797.607124
8,2090.0,3210.0,1120.0,615.187988,4810.0,2450.0,1470.0,974.772363,1390.0,911.550998
9,2390.0,3430.0,1280.0,692.086486,5500.0,2800.0,1540.0,1140.0,1580.0,1030.0


In [7]:
cols = [
    "KH-SG",
    "MY-SG",
    "ID-SG",
    "SG-VN",
]
data[cols]

,KH-SG,MY-SG,ID-SG,SG-VN
0,0.0,0.0,0.0,0.0
1,162.46206,418.395294,472.035669,274.937286
2,324.924121,585.50141,944.071337,549.874572
3,405.384283,836.790589,1420.0,824.811857
4,487.386181,1170.0,1890.0,877.549283
5,649.848242,1260.0,2220.0,1100.0
6,810.768566,1670.0,2360.0,1370.0
7,812.310302,1760.0,2830.0,1650.0
8,974.772363,2090.0,3300.0,1760.0
9,1140.0,2340.0,3780.0,1920.0


In [8]:
for country in country_emissions:
    case_study = country
    if case_study not in existing_cases or len(existing_cases[case_study]) < 11:
        print(f"Generating parameters for: {country}")
        values = process_combination([country], country_emissions)
        type_index = write_parameters(output_file, case_study, values, type_index)
    elif case_study in existing_cases:
         print(f"Updating parameters for: {country}")
         values = process_combination([country], country_emissions)
         updated = update_existing_case_values(output_file, case_study, values)

Updating parameters for: AU
Updating parameters for: RU
Updating parameters for: PH
Updating parameters for: PE
Updating parameters for: LA
Updating parameters for: BR
Updating parameters for: US
Updating parameters for: CO
Updating parameters for: EG
Updating parameters for: NG
Updating parameters for: ZA
Updating parameters for: JP
Updating parameters for: MA
Updating parameters for: VN
Updating parameters for: MY
Updating parameters for: FR
Updating parameters for: ID
Updating parameters for: KR
Updating parameters for: SG
Updating parameters for: KH
Updating parameters for: CL
Updating parameters for: IN
Updating parameters for: CN
Updating parameters for: BN
Updating parameters for: MM
Updating parameters for: TR
Updating parameters for: DE
Updating parameters for: SA
Updating parameters for: KE
Updating parameters for: TH


In [9]:
cols[0].split("-")

['KH', 'SG']

In [10]:
combos = list(map(lambda x: x.split("-"), cols))

for combo in combos:
    if not combo:
        pass
    if len(combo) > 4:
        print("⚠️ Please enter no more than 4 countries for collaboration.")
    else:
        # Generate all unique combinations of size 2 to len(combo)
        for r in range(2, len(combo) + 1):
            for sub_combo in itertools.combinations(combo, r):
                combo_sorted = sorted(sub_combo)
                case_study = "-".join(combo_sorted)
    
                if case_study in existing_cases:
                    print(f"Updating collaboration: {case_study}")
                    try:
                        values = process_combination(combo_sorted, country_emissions)
                        updated = update_existing_case_values(output_file, case_study, values)
                    except KeyError as e:
                        print(f"⚠️ Country not found in emissions data: {e}")
                else:
                    try:
                        print(f"Generating collaboration: {case_study}")
                        type_index = write_parameters(output_file, case_study, values, type_index)
                    except KeyError as e:
                        print(f"⚠️ Country not found in emissions data: {e}")

Updating collaboration: KH-SG
Updating collaboration: MY-SG
Updating collaboration: ID-SG
Updating collaboration: SG-VN


In [12]:
data[cols]

,KH-SG,MY-SG,ID-SG,SG-VN
0,0.0,0.0,0.0,0.0
1,162.46206,418.395294,472.035669,274.937286
2,324.924121,585.50141,944.071337,549.874572
3,405.384283,836.790589,1420.0,824.811857
4,487.386181,1170.0,1890.0,877.549283
5,649.848242,1260.0,2220.0,1100.0
6,810.768566,1670.0,2360.0,1370.0
7,812.310302,1760.0,2830.0,1650.0
8,974.772363,2090.0,3300.0,1760.0
9,1140.0,2340.0,3780.0,1920.0


In [14]:
values

[0.0,
 274.9372858211136,
 549.8745716422272,
 824.8118574633409,
 877.5492827660288,
 1099.7491432844545,
 1374.686429105568,
 1649.6237149266817,
 1755.0985655320576,
 1924.5610007477953,
 2199.498286568909]